# Stage 02 · Step 0 — Generate multi-τ training corpus

The supervised RUL head benefits from telemetry observed under several maintenance schedules, not just the single baseline τ already in `data/fleet_baseline.parquet`. This notebook draws K τ vectors via Latin-Hypercube sampling over the same ranges Stage 01 explores and runs the SDG simulator on the **train printers (id 0..69)** for each.

Outputs:
- `data/policy_runs/policy_{k:03d}.parquet` — one parquet per τ schedule (train printers only, RUL labels included).
- `data/policy_runs/manifest.json` — index of τ values per file.

Skip this notebook if the corpus already exists; the SSL pretraining notebook will fall back to `fleet_baseline.parquet` if no policy runs are found.

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
from scipy.stats import qmc

from ml_models.lib.data import TRAIN_PRINTERS
from ml_models.lib.env_runner import default_dates, run_with_tau
from sdg.generate import load_configs
from sdg.labels import add_rul_labels
from sdg.schema import COMPONENT_IDS, table_from_dataframe

POLICY_DIR = Path('data/policy_runs')
POLICY_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = POLICY_DIR / 'manifest.json'

In [ ]:
TAU_RANGES = {
    'C1': (50.0, 2_000.0),
    'C2': (500.0, 20_000.0),
    'C3': (24.0, 500.0),
    'C4': (100.0, 2_000.0),
    'C5': (500.0, 8_000.0),
    'C6': (1_000.0, 20_000.0),
}
K = 5  # number of policy runs; set higher (e.g. 30) for stronger surrogate generalisation
SEED = 17

sampler = qmc.LatinHypercube(d=len(TAU_RANGES), seed=SEED)
unit = sampler.random(K)
tau_schedules: list[dict[str, float]] = []
for row in unit:
    schedule = {}
    for i, (component_id, (low, high)) in enumerate(TAU_RANGES.items()):
        schedule[component_id] = float(np.exp(np.log(low) + row[i] * (np.log(high) - np.log(low))))
    tau_schedules.append(schedule)
tau_schedules

In [ ]:
components_cfg, couplings_cfg, cities_cfg = load_configs()
DATES = default_dates()
manifest = []
for k, schedule in enumerate(tau_schedules):
    output_path = POLICY_DIR / f'policy_{k:03d}.parquet'
    if output_path.exists():
        print(f'[{k}] cached:', output_path)
        manifest.append({'k': k, 'tau': schedule, 'path': output_path.as_posix()})
        continue
    print(f'[{k}] simulating', schedule)
    df = run_with_tau(
        schedule,
        printer_ids=TRAIN_PRINTERS,
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    table = table_from_dataframe(df, include_rul=False)
    pq.write_table(table, output_path, compression='snappy', version='2.6')
    add_rul_labels(output_path)
    manifest.append({'k': k, 'tau': schedule, 'path': output_path.as_posix()})
    print(f'[{k}] wrote {output_path} ({output_path.stat().st_size / 1e6:.1f} MB)')

In [ ]:
with MANIFEST_PATH.open('w', encoding='utf-8') as handle:
    json.dump({'tau_ranges': TAU_RANGES, 'seed': SEED, 'runs': manifest}, handle, indent=2)
print('Wrote manifest:', MANIFEST_PATH)
print(f'{len(manifest)} policy runs covering printer_ids {TRAIN_PRINTERS[0]}..{TRAIN_PRINTERS[-1]}')